# Quant LongTerm AI – Research Notebook
Interactive exploration of data, features, model outputs and backtest results.

In [ ]:
import sys
sys.path.insert(0, '..')   # make src/ importable

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['axes.grid'] = True

from src.utils import load_config, get_logger
log = get_logger('research')
cfg = load_config('../config.yaml')
print('Config loaded ✓')

## 1. Data Ingestion

In [ ]:
from src.data_ingestion import DataIngestion
from src.data_validation import DataValidator

ing = DataIngestion(cfg)
raw = ing.fetch_all()
print(f'Fetched {len(raw)} tickers')
list(raw.keys())[:5]

In [ ]:
# Inspect one ticker
tkr = 'TCS.NS'
df = raw.get(tkr)
if df is not None:
    print(df.tail())
    df['Close'].plot(title=f'{tkr} Close Price')
    plt.tight_layout(); plt.show()

In [ ]:
val = DataValidator(cfg)
clean = val.validate(raw)
aligned = val.align_panel(clean)

benchmark_ticker = cfg['data']['benchmark_ticker']
stock_data = {k: v for k, v in aligned.items() if k != benchmark_ticker}
benchmark_df = clean.get(benchmark_ticker)
print(f'Validated tickers: {len(stock_data)}')

## 2. Feature Engineering

In [ ]:
from src.feature_engineering import FeatureEngineer

fe = FeatureEngineer(cfg)
panel = fe.build_features(stock_data)
panel = fe.add_cross_sectional_features(panel)
print(f'Feature panel shape: {panel.shape}')
panel.head(3)

In [ ]:
# Correlation heatmap of features for one ticker
import seaborn as sns

tkr_panel = panel.xs(tkr, level='Ticker') if tkr in panel.index.get_level_values('Ticker') else panel.groupby(level='Ticker').first()
feat_cols = [c for c in tkr_panel.columns if tkr_panel[c].dtype in ['float64','float32']]
corr = tkr_panel[feat_cols[:20]].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='RdBu_r', center=0, annot=False, linewidths=0.3)
plt.title('Feature Correlation Matrix (first 20 features)')
plt.tight_layout(); plt.show()

## 3. Alpha Factors

In [ ]:
from src.alpha_factors import AlphaFactors
from src.pipeline import QuantPipeline

# Merge OHLCV into panel for alpha computation
pipeline_obj = QuantPipeline(config_path='../config.yaml')
panel_with_ohlcv = pipeline_obj._merge_ohlcv(panel, stock_data)

af = AlphaFactors(cfg)
panel_alpha = af.compute(panel_with_ohlcv)

alpha_cols = [c for c in panel_alpha.columns if c.startswith('a0')]
print(f'Alpha factors computed: {len(alpha_cols)}')

In [ ]:
# Alpha IC (Information Coefficient) analysis
from src.labeling import Labeler

labeler = Labeler(cfg)
labeled = labeler.label(panel_alpha, benchmark_df)

# Compute IC for each alpha factor
fwd_col = 'fwd_ret_primary'
ic_results = {}
for col in alpha_cols[:30]:
    try:
        ic = labeled[[col, fwd_col]].dropna().corr().iloc[0, 1]
        ic_results[col] = ic
    except:
        pass

ic_series = pd.Series(ic_results).sort_values(ascending=False)
print('Top 10 IC factors:')
print(ic_series.head(10))

ic_series.plot(kind='bar', title='Alpha Factor IC vs Primary Horizon', figsize=(14, 4))
plt.axhline(0, color='black', linewidth=0.8)
plt.ylabel('Information Coefficient')
plt.tight_layout(); plt.show()

## 4. Model Training & Feature Importance

In [ ]:
from src.model import ModelTrainer

X, y, feature_cols = pipeline_obj._prepare_ml_data(labeled)
print(f'X shape: {X.shape} | Positive rate: {y.mean():.2%}')

trainer = ModelTrainer(cfg)
models = trainer.train(X, y)

In [ ]:
# Feature importance plot
for name, model in models.items():
    try:
        imp = model.feature_importances().head(20)
        imp.sort_values().plot(kind='barh', title=f'{name} – Top 20 Features', figsize=(10, 6))
        plt.tight_layout(); plt.show()
    except Exception as e:
        print(f'Could not plot {name}: {e}')

## 5. Ensemble Scoring & Ranking

In [ ]:
from src.ensemble import EnsemblePredictor
from src.ranking import StockRanker

ep = EnsemblePredictor(cfg)
preds = trainer.predict(models, X)
scores = ep.predict(preds, index=X.index)

ranker = StockRanker(cfg)
ranked = ranker.rank(scores)

latest_top = ranker.latest_top_stocks(scores)
print('\nTop stocks:')
print(latest_top)

In [ ]:
# Score distribution
scores.hist(bins=50, title='Ensemble Score Distribution', edgecolor='black')
plt.xlabel('Score'); plt.ylabel('Count')
plt.tight_layout(); plt.show()

In [ ]:
# Stock selection frequency
summary = ranker.score_summary(ranked)
summary.head(10)

## 6. Portfolio Optimization

In [ ]:
from src.portfolio import PortfolioOptimizer

po = PortfolioOptimizer(cfg)
top_tickers = latest_top['Ticker'].tolist()
weights = po.optimize(top_tickers, stock_data)

print('Portfolio Weights:')
for tkr, w in sorted(weights.items(), key=lambda x: -x[1]):
    print(f'  {tkr:20s}: {w:.2%}')

# Pie chart
plt.figure(figsize=(8,8))
labels = [f"{k}\n{v:.1%}" for k, v in weights.items() if v > 0.001]
vals = [v for v in weights.values() if v > 0.001]
plt.pie(vals, labels=labels, autopct='%1.1f%%', startangle=90)
plt.title('Portfolio Allocation', fontsize=13)
plt.tight_layout(); plt.show()

## 7. Backtest Results

In [ ]:
# Load backtest results if pipeline was run
import os
metrics_path = '../results/metrics.csv'
if os.path.exists(metrics_path):
    metrics = pd.read_csv(metrics_path)
    print(metrics.T)
else:
    print('Run the full pipeline first: python main.py')

In [ ]:
# Display saved charts
import matplotlib.image as mpimg

charts = ['../results/equity_curve.png', '../results/drawdown.png',
          '../results/rolling_metrics.png', '../results/monthly_heatmap.png']

for chart in charts:
    if os.path.exists(chart):
        img = mpimg.imread(chart)
        plt.figure(figsize=(14, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.title(os.path.basename(chart))
        plt.tight_layout(); plt.show()
    else:
        print(f'Chart not found: {chart}')

## 8. Factor Analysis
Quintile analysis of key alpha factors

In [ ]:
# Quintile return analysis for top IC factor
if ic_results:
    best_factor = ic_series.index[0]
    analysis_df = labeled[[best_factor, fwd_col]].dropna().copy()
    analysis_df['quintile'] = pd.qcut(analysis_df[best_factor], 5, labels=['Q1','Q2','Q3','Q4','Q5'])
    quintile_returns = analysis_df.groupby('quintile')[fwd_col].mean() * 100

    quintile_returns.plot(
        kind='bar',
        title=f'Quintile Returns: {best_factor}',
        color=['#d32f2f','#ef5350','#bdbdbd','#66bb6a','#2e7d32'],
        figsize=(8, 4)
    )
    plt.axhline(0, color='black', linewidth=0.8)
    plt.ylabel('Avg Forward Return (%)')
    plt.xticks(rotation=0)
    plt.tight_layout(); plt.show()
    print(quintile_returns)